# Notebook 26 — DINOv2-S Clean Representation Audit

## Objective

Previous experiments with the ResNet18-based student showed that increasingly sophisticated cross-script objectives produced only limited gains. In particular, WGC improved the baseline, while subsequent teacher-guided approaches did not produce a reliable additional improvement.

This notebook therefore tests a more fundamental hypothesis:

> Is the main limitation of the current student caused by the representation itself rather than by the training objective?

To test this cleanly, this notebook evaluates a stronger frozen representation from **DINOv2-S** without introducing a new end-to-end neural training procedure.

## Representation Pipeline

The DINOv2-S features used here come from the previously cached `dinov2_vits14_reg` feature archive.

The original DINOv2-S extraction pipeline:

1. resized each handwriting image to the DINOv2 input resolution,
2. applied ImageNet normalization,
3. extracted a 384-dimensional DINOv2-S feature,
4. immediately L2-normalized the extracted feature,
5. stored the normalized feature in the cached archive.

Therefore, the input to the present PCA/LDA audit is not an unnormalized raw backbone output. It is a **cached, already L2-normalized 384-D DINOv2-S feature**.

The adaptation pipeline evaluated in this notebook is:

**cached normalized DINOv2-S 384-D feature → PCA(225) → writer-LDA → L2 normalization → cosine verification**

PCA and writer-LDA are fitted only on the **145 development-fit writers**. The fitted transformation is then applied unchanged to the separate **36-writer selection split**.

The final LDA dimensionality is determined by the available writer-discriminant rank and is 144 dimensions for the 145-writer fit set.

## Experimental Protocol

This notebook follows the clean internal development protocol:

- Fit writers: **145**
- Selection writers: **36**
- Fit and selection writers are disjoint
- Four pages are available per writer
- Evaluation uses the same four cross-script verification conditions used in the previous controlled experiments
- No monitor writers are used
- No validation split is used
- No official test split is used

The same cross-script pair protocol and cosine-scoring evaluator are retained so that the result can be compared directly with the previous ResNet18 student experiments.

## Interpretation Goal

This experiment is not intended to establish the final trainable student architecture.

Instead, it asks whether substantially stronger cross-script writer information already exists in a better frozen representation.

A large improvement over the ResNet18-based WGC result would support a **representation-bottleneck hypothesis** and justify moving the student architecture toward DINOv2-S.

A small improvement would instead suggest that the remaining limitation lies primarily in the downstream learning objective or verification formulation.

In [1]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score, roc_curve

In [3]:
ROOT = Path(
    "/home/arijit/Documents/handwriting-cross-script-research"
)

SPLIT_PATH = (
    ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

ROLE_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "development_internal_writer_roles_seed42.csv"
)

DINO_REPORT_DIR = (
    ROOT
    / "reports"
    / "modern_baseline_benchmarking"
)

split_df = pd.read_csv(
    SPLIT_PATH
)

role_df = pd.read_csv(
    ROLE_PATH
)

role_column_candidates = []

for column in role_df.columns:
    values = set(
        role_df[
            column
        ]
        .astype(str)
        .str.lower()
        .unique()
    )

    if {
        "fit",
        "selection",
    }.issubset(
        values
    ):
        role_column_candidates.append(
            column
        )

if len(
    role_column_candidates
) != 1:
    print(
        "Role-file columns:",
        list(
            role_df.columns
        ),
    )

    print(
        "\nRole-file preview:"
    )

    print(
        role_df
        .head()
        .to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Could not uniquely identify the fit/selection role column."
    )

ROLE_COLUMN = (
    role_column_candidates[
        0
    ]
)

dino_npz_candidates = sorted(
    [
        path
        for path in DINO_REPORT_DIR.rglob(
            "*.npz"
        )
        if "dino" in path.name.lower()
    ]
)

resource_audit = {
    "root_exists": bool(
        ROOT.exists()
    ),
    "split_path_exists": bool(
        SPLIT_PATH.exists()
    ),
    "role_path_exists": bool(
        ROLE_PATH.exists()
    ),
    "dino_report_dir_exists": bool(
        DINO_REPORT_DIR.exists()
    ),
    "split_rows": int(
        len(
            split_df
        )
    ),
    "split_columns": list(
        split_df.columns
    ),
    "role_rows": int(
        len(
            role_df
        )
    ),
    "role_columns": list(
        role_df.columns
    ),
    "detected_role_column": (
        ROLE_COLUMN
    ),
    "dino_npz_candidate_count": int(
        len(
            dino_npz_candidates
        )
    ),
    "dino_npz_candidates": [
        str(
            path.relative_to(
                ROOT
            )
        )
        for path in (
            dino_npz_candidates
        )
    ],
}

print(
    json.dumps(
        resource_audit,
        indent=2,
    )
)

print(
    "\nRole counts:"
)

print(
    role_df[
        ROLE_COLUMN
    ]
    .value_counts()
    .to_string()
)

print(
    "\nDINO embedding archives:"
)

for path in dino_npz_candidates:
    archive = np.load(
        path,
        allow_pickle=False,
    )

    print(
        f"\n{path.name}"
    )

    for key in archive.files:
        array = archive[
            key
        ]

        print(
            f"  {key}: "
            f"shape={array.shape}, "
            f"dtype={array.dtype}"
        )

    archive.close()

if (
    not SPLIT_PATH.exists()
    or not ROLE_PATH.exists()
    or len(
        role_df
    ) != 724
):
    raise RuntimeError(
        "Notebook 26 resource audit failed."
    )

{
  "root_exists": true,
  "split_path_exists": true,
  "role_path_exists": true,
  "dino_report_dir_exists": true,
  "split_rows": 1900,
  "split_columns": [
    "filename",
    "writer",
    "page_id",
    "language",
    "same_text",
    "source_split",
    "experiment_split"
  ],
  "role_rows": 724,
  "role_columns": [
    "filename",
    "writer",
    "page_id",
    "language",
    "same_text",
    "internal_role"
  ],
  "detected_role_column": "internal_role",
  "dino_npz_candidate_count": 2,
  "dino_npz_candidates": [
    "reports/modern_baseline_benchmarking/dinov2_vitg14_reg_embeddings.npz",
    "reports/modern_baseline_benchmarking/dinov2_vitl14_reg_embeddings.npz"
  ]
}

Role counts:
internal_role
fit          580
selection    144

DINO embedding archives:

dinov2_vitg14_reg_embeddings.npz
  development_embeddings: shape=(904, 1536), dtype=float32
  development_filenames: shape=(904,), dtype=<U10
  development_writers: shape=(904,), dtype=int64
  validation_embeddings: shape

In [4]:
SEARCH_ROOTS = [
    ROOT / "notebooks",
    ROOT / "reports",
    ROOT / "src",
]

DINO_S_TERMS = [
    "dinov2_vits14_reg",
    "dinov2_vits14",
    "DINOv2-S",
    "dinov2-s",
    "vits14_reg",
    "vits14",
]

candidate_files = []

for search_root in SEARCH_ROOTS:
    if not search_root.exists():
        continue

    for path in search_root.rglob(
        "*"
    ):
        if not path.is_file():
            continue

        filename_lower = (
            path.name.lower()
        )

        if any(
            term.lower()
            in filename_lower
            for term in (
                DINO_S_TERMS
            )
        ):
            candidate_files.append(
                {
                    "path": str(
                        path.relative_to(
                            ROOT
                        )
                    ),
                    "match_source": (
                        "filename"
                    ),
                }
            )

text_extensions = {
    ".py",
    ".ipynb",
    ".md",
    ".txt",
    ".json",
    ".csv",
}

text_match_rows = []

for search_root in SEARCH_ROOTS:
    if not search_root.exists():
        continue

    for path in search_root.rglob(
        "*"
    ):
        if (
            not path.is_file()
            or path.suffix.lower()
            not in text_extensions
        ):
            continue

        try:
            text = path.read_text(
                encoding="utf-8"
            )
        except Exception:
            continue

        matched_terms = [
            term
            for term in (
                DINO_S_TERMS
            )
            if term.lower()
            in text.lower()
        ]

        if len(
            matched_terms
        ) > 0:
            text_match_rows.append(
                {
                    "path": str(
                        path.relative_to(
                            ROOT
                        )
                    ),
                    "matched_terms": (
                        ", ".join(
                            matched_terms
                        )
                    ),
                }
            )

binary_extensions = {
    ".npz",
    ".npy",
    ".pt",
    ".pth",
    ".ckpt",
}

binary_candidate_rows = []

for search_root in [
    ROOT / "reports",
    ROOT / "checkpoints",
]:
    if not search_root.exists():
        continue

    for path in search_root.rglob(
        "*"
    ):
        if (
            path.is_file()
            and path.suffix.lower()
            in binary_extensions
        ):
            filename_lower = (
                path.name.lower()
            )

            if any(
                token
                in filename_lower
                for token in [
                    "dino",
                    "vits",
                ]
            ):
                binary_candidate_rows.append(
                    {
                        "path": str(
                            path.relative_to(
                                ROOT
                            )
                        ),
                        "size_mb": float(
                            path.stat().st_size
                            / (
                                1024
                                ** 2
                            )
                        ),
                    }
                )

dino_s_artifact_audit = {
    "filename_matches": int(
        len(
            candidate_files
        )
    ),
    "text_files_with_dinov2s_reference": int(
        len(
            text_match_rows
        )
    ),
    "dino_binary_candidates": int(
        len(
            binary_candidate_rows
        )
    ),
    "existing_modern_benchmark_archives": [
        str(
            path.relative_to(
                ROOT
            )
        )
        for path in (
            dino_npz_candidates
        )
    ],
    "dinov2s_modern_benchmark_archive_present": bool(
        any(
            "vits"
            in path.name.lower()
            for path in (
                dino_npz_candidates
            )
        )
    ),
    "pca_or_evaluation_performed": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

print(
    json.dumps(
        dino_s_artifact_audit,
        indent=2,
    )
)

print(
    "\nDINOv2-S filename matches:"
)

if candidate_files:
    print(
        pd.DataFrame(
            candidate_files
        ).to_string(
            index=False
        )
    )
else:
    print(
        "None"
    )

print(
    "\nFiles containing DINOv2-S references:"
)

if text_match_rows:
    print(
        pd.DataFrame(
            text_match_rows
        ).to_string(
            index=False
        )
    )
else:
    print(
        "None"
    )

print(
    "\nDINO-related binary candidates:"
)

if binary_candidate_rows:
    print(
        pd.DataFrame(
            binary_candidate_rows
        )
        .sort_values(
            "path"
        )
        .round(
            {
                "size_mb": 3
            }
        )
        .to_string(
            index=False
        )
    )
else:
    print(
        "None"
    )

{
  "filename_matches": 1,
  "text_files_with_dinov2s_reference": 4,
  "dino_binary_candidates": 4,
  "existing_modern_benchmark_archives": [
    "reports/modern_baseline_benchmarking/dinov2_vitg14_reg_embeddings.npz",
    "reports/modern_baseline_benchmarking/dinov2_vitl14_reg_embeddings.npz"
  ],
  "dinov2s_modern_benchmark_archive_present": false,
  "pca_or_evaluation_performed": false,
  "selection_evaluated": false,
  "monitor_used": false,
  "validation_used": false,
  "official_test_used": false
}

DINOv2-S filename matches:
                                                               path match_source
reports/dinov2_efficiency_frontier/dinov2_vits14_reg_embeddings.npz     filename

Files containing DINOv2-S references:
                                                                            path                                                            matched_terms
                                   notebooks/20_dinov2_efficiency_frontier.ipynb dinov2_vits14_reg, dinov2_

In [5]:
DINO_S_PATH = (
    ROOT
    / "reports"
    / "dinov2_efficiency_frontier"
    / "dinov2_vits14_reg_embeddings.npz"
)

REPORT_DIR = (
    ROOT
    / "reports"
    / "dinov2s_clean_representation_audit"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

dino_s_archive = np.load(
    DINO_S_PATH,
    allow_pickle=False,
)

print(
    "DINOv2-S archive:"
)

for key in dino_s_archive.files:
    array = dino_s_archive[
        key
    ]

    print(
        f"{key}: "
        f"shape={array.shape}, "
        f"dtype={array.dtype}"
    )

if (
    "development_embeddings"
    not in dino_s_archive.files
    or "development_filenames"
    not in dino_s_archive.files
):
    raise RuntimeError(
        "Expected raw development DINOv2-S arrays were not found."
    )

dino_s_development_embeddings = (
    dino_s_archive[
        "development_embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

dino_s_development_filenames = (
    dino_s_archive[
        "development_filenames"
    ]
    .astype(str)
)

if (
    "development_writers"
    in dino_s_archive.files
):
    dino_s_development_writers = (
        dino_s_archive[
            "development_writers"
        ]
        .astype(
            np.int64,
            copy=False,
        )
    )
else:
    dino_s_development_writers = None

if (
    dino_s_development_embeddings.ndim != 2
    or dino_s_development_embeddings.shape[
        1
    ] != 384
):
    raise RuntimeError(
        "DINOv2-S raw development embeddings are not 384-D."
    )

if (
    len(
        dino_s_development_filenames
    )
    != len(
        dino_s_development_embeddings
    )
):
    raise RuntimeError(
        "DINOv2-S filename and embedding counts do not match."
    )

if (
    len(
        np.unique(
            dino_s_development_filenames
        )
    )
    != len(
        dino_s_development_filenames
    )
):
    raise RuntimeError(
        "Duplicate filenames found in DINOv2-S development archive."
    )

filename_to_dino_s_index = {
    filename: int(
        index
    )
    for index, filename in enumerate(
        dino_s_development_filenames
    )
}

fit_dino_s_rows = (
    role_df[
        role_df[
            ROLE_COLUMN
        ] == "fit"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

selection_dino_s_rows = (
    role_df[
        role_df[
            ROLE_COLUMN
        ] == "selection"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

fit_missing_filenames = [
    filename
    for filename in (
        fit_dino_s_rows[
            "filename"
        ].astype(str)
    )
    if filename
    not in filename_to_dino_s_index
]

selection_missing_filenames = [
    filename
    for filename in (
        selection_dino_s_rows[
            "filename"
        ].astype(str)
    )
    if filename
    not in filename_to_dino_s_index
]

if (
    len(
        fit_missing_filenames
    ) > 0
    or len(
        selection_missing_filenames
    ) > 0
):
    raise RuntimeError(
        "Fit or selection filenames are missing from the DINOv2-S archive."
    )

fit_dino_s_indices = np.array(
    [
        filename_to_dino_s_index[
            filename
        ]
        for filename in (
            fit_dino_s_rows[
                "filename"
            ].astype(str)
        )
    ],
    dtype=np.int64,
)

selection_dino_s_indices = np.array(
    [
        filename_to_dino_s_index[
            filename
        ]
        for filename in (
            selection_dino_s_rows[
                "filename"
            ].astype(str)
        )
    ],
    dtype=np.int64,
)

fit_dino_s_raw = (
    dino_s_development_embeddings[
        fit_dino_s_indices
    ]
)

selection_dino_s_raw = (
    dino_s_development_embeddings[
        selection_dino_s_indices
    ]
)

writer_mismatch_count = 0

if (
    dino_s_development_writers
    is not None
):
    fit_archive_writers = (
        dino_s_development_writers[
            fit_dino_s_indices
        ]
    )

    selection_archive_writers = (
        dino_s_development_writers[
            selection_dino_s_indices
        ]
    )

    writer_mismatch_count = int(
        (
            fit_archive_writers
            != fit_dino_s_rows[
                "writer"
            ].to_numpy(
                dtype=np.int64
            )
        ).sum()
        + (
            selection_archive_writers
            != selection_dino_s_rows[
                "writer"
            ].to_numpy(
                dtype=np.int64
            )
        ).sum()
    )

fit_page_counts = (
    fit_dino_s_rows
    .groupby(
        "writer"
    )
    .size()
)

selection_page_counts = (
    selection_dino_s_rows
    .groupby(
        "writer"
    )
    .size()
)

fit_selection_writer_overlap = (
    set(
        fit_dino_s_rows[
            "writer"
        ].astype(int)
    )
    & set(
        selection_dino_s_rows[
            "writer"
        ].astype(int)
    )
)

dino_s_raw_alignment_audit = {
    "artifact_path": str(
        DINO_S_PATH.relative_to(
            ROOT
        )
    ),
    "development_embedding_shape": list(
        dino_s_development_embeddings.shape
    ),
    "raw_embedding_dimension": int(
        dino_s_development_embeddings.shape[
            1
        ]
    ),
    "development_filenames": int(
        len(
            dino_s_development_filenames
        )
    ),
    "fit_pages": int(
        len(
            fit_dino_s_rows
        )
    ),
    "fit_writers": int(
        fit_dino_s_rows[
            "writer"
        ].nunique()
    ),
    "fit_embedding_shape": list(
        fit_dino_s_raw.shape
    ),
    "selection_pages": int(
        len(
            selection_dino_s_rows
        )
    ),
    "selection_writers": int(
        selection_dino_s_rows[
            "writer"
        ].nunique()
    ),
    "selection_embedding_shape": list(
        selection_dino_s_raw.shape
    ),
    "fit_missing_filenames": int(
        len(
            fit_missing_filenames
        )
    ),
    "selection_missing_filenames": int(
        len(
            selection_missing_filenames
        )
    ),
    "writer_mismatch_count": int(
        writer_mismatch_count
    ),
    "fit_selection_writer_overlap": int(
        len(
            fit_selection_writer_overlap
        )
    ),
    "fit_four_pages_per_writer": bool(
        (
            fit_page_counts
            == 4
        ).all()
    ),
    "selection_four_pages_per_writer": bool(
        (
            selection_page_counts
            == 4
        ).all()
    ),
    "fit_language_counts": {
        str(
            key
        ): int(
            value
        )
        for key, value in (
            fit_dino_s_rows[
                "language"
            ]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "selection_language_counts": {
        str(
            key
        ): int(
            value
        )
        for key, value in (
            selection_dino_s_rows[
                "language"
            ]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "raw_embeddings_reused_without_reextraction": True,
    "raw_l2_normalization_performed": False,
    "pca_performed": False,
    "lda_performed": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "dinov2s_raw_alignment_audit.json",
    "w",
) as file:
    json.dump(
        dino_s_raw_alignment_audit,
        file,
        indent=2,
    )

print(
    "\nRaw alignment audit:"
)

print(
    json.dumps(
        dino_s_raw_alignment_audit,
        indent=2,
    )
)

if (
    dino_s_development_embeddings.shape
    != (
        904,
        384,
    )
    or len(
        fit_dino_s_rows
    ) != 580
    or fit_dino_s_rows[
        "writer"
    ].nunique()
    != 145
    or fit_dino_s_raw.shape
    != (
        580,
        384,
    )
    or len(
        selection_dino_s_rows
    ) != 144
    or selection_dino_s_rows[
        "writer"
    ].nunique()
    != 36
    or selection_dino_s_raw.shape
    != (
        144,
        384,
    )
    or len(
        fit_missing_filenames
    ) != 0
    or len(
        selection_missing_filenames
    ) != 0
    or writer_mismatch_count
    != 0
    or len(
        fit_selection_writer_overlap
    ) != 0
    or not (
        fit_page_counts
        == 4
    ).all()
    or not (
        selection_page_counts
        == 4
    ).all()
):
    raise RuntimeError(
        "DINOv2-S clean raw alignment failed."
    )

DINOv2-S archive:
development_embeddings: shape=(904, 384), dtype=float32
development_filenames: shape=(904,), dtype=<U10
development_writers: shape=(904,), dtype=int64
validation_embeddings: shape=(224, 384), dtype=float32
validation_filenames: shape=(224,), dtype=<U10

Raw alignment audit:
{
  "artifact_path": "reports/dinov2_efficiency_frontier/dinov2_vits14_reg_embeddings.npz",
  "development_embedding_shape": [
    904,
    384
  ],
  "raw_embedding_dimension": 384,
  "development_filenames": 904,
  "fit_pages": 580,
  "fit_writers": 145,
  "fit_embedding_shape": [
    580,
    384
  ],
  "selection_pages": 144,
  "selection_writers": 36,
  "selection_embedding_shape": [
    144,
    384
  ],
  "fit_missing_filenames": 0,
  "selection_missing_filenames": 0,
  "writer_mismatch_count": 0,
  "fit_selection_writer_overlap": 0,
  "fit_four_pages_per_writer": true,
  "selection_four_pages_per_writer": true,
  "fit_language_counts": {
    "Arabic": 290,
    "English": 290
  },
  "selecti

In [6]:
DINO_S_PCA_COMPONENTS = 225

dino_s_pca = PCA(
    n_components=DINO_S_PCA_COMPONENTS,
    svd_solver="full",
)

fit_dino_s_pca = (
    dino_s_pca.fit_transform(
        fit_dino_s_raw
    )
)

selection_dino_s_pca = (
    dino_s_pca.transform(
        selection_dino_s_raw
    )
)

fit_writer_labels = (
    fit_dino_s_rows[
        "writer"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

dino_s_lda = (
    LinearDiscriminantAnalysis(
        solver="svd"
    )
)

fit_dino_s_lda_raw = (
    dino_s_lda.fit_transform(
        fit_dino_s_pca,
        fit_writer_labels,
    )
)

selection_dino_s_lda_raw = (
    dino_s_lda.transform(
        selection_dino_s_pca
    )
)


def l2_normalize_rows(
    array,
):
    norms = np.linalg.norm(
        array,
        axis=1,
        keepdims=True,
    )

    if (
        not np.isfinite(
            norms
        ).all()
        or (
            norms
            <= 0.0
        ).any()
    ):
        raise RuntimeError(
            "Invalid embedding norm before L2 normalization."
        )

    return (
        array
        / norms
    )


fit_dino_s_adapted = (
    l2_normalize_rows(
        fit_dino_s_lda_raw
    )
    .astype(
        np.float32,
        copy=False,
    )
)

selection_dino_s_adapted = (
    l2_normalize_rows(
        selection_dino_s_lda_raw
    )
    .astype(
        np.float32,
        copy=False,
    )
)

fit_adapted_norms = np.linalg.norm(
    fit_dino_s_adapted,
    axis=1,
)

selection_adapted_norms = np.linalg.norm(
    selection_dino_s_adapted,
    axis=1,
)

dino_s_transformation_audit = {
    "representation": "DINOv2-S/14-Reg",
    "raw_dimension": int(
        fit_dino_s_raw.shape[
            1
        ]
    ),
    "raw_l2_normalization_before_pca": False,
    "pca_components": int(
        DINO_S_PCA_COMPONENTS
    ),
    "pca_solver": "full",
    "pca_fit_pages": int(
        len(
            fit_dino_s_raw
        )
    ),
    "pca_fit_writers": int(
        fit_dino_s_rows[
            "writer"
        ].nunique()
    ),
    "pca_selection_used_for_fit": False,
    "pca_retained_variance": float(
        dino_s_pca
        .explained_variance_ratio_
        .sum()
    ),
    "fit_pca_shape": list(
        fit_dino_s_pca.shape
    ),
    "selection_pca_shape": list(
        selection_dino_s_pca.shape
    ),
    "lda_solver": "svd",
    "lda_fit_pages": int(
        len(
            fit_dino_s_pca
        )
    ),
    "lda_fit_writers": int(
        len(
            np.unique(
                fit_writer_labels
            )
        )
    ),
    "lda_selection_used_for_fit": False,
    "lda_output_dimension": int(
        fit_dino_s_lda_raw.shape[
            1
        ]
    ),
    "fit_lda_shape": list(
        fit_dino_s_lda_raw.shape
    ),
    "selection_lda_shape": list(
        selection_dino_s_lda_raw.shape
    ),
    "post_lda_l2_normalization": True,
    "fit_adapted_shape": list(
        fit_dino_s_adapted.shape
    ),
    "selection_adapted_shape": list(
        selection_dino_s_adapted.shape
    ),
    "fit_norm_min": float(
        fit_adapted_norms.min()
    ),
    "fit_norm_max": float(
        fit_adapted_norms.max()
    ),
    "selection_norm_min": float(
        selection_adapted_norms.min()
    ),
    "selection_norm_max": float(
        selection_adapted_norms.max()
    ),
    "selection_labels_used_for_transformation_fit": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

np.savez_compressed(
    REPORT_DIR
    / "dinov2s_fit_selection_adapted_embeddings.npz",
    fit_embeddings=(
        fit_dino_s_adapted
    ),
    fit_filenames=(
        fit_dino_s_rows[
            "filename"
        ]
        .astype(str)
        .to_numpy()
    ),
    fit_writers=(
        fit_dino_s_rows[
            "writer"
        ]
        .to_numpy(
            dtype=np.int64
        )
    ),
    fit_page_ids=(
        fit_dino_s_rows[
            "page_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    ),
    selection_embeddings=(
        selection_dino_s_adapted
    ),
    selection_filenames=(
        selection_dino_s_rows[
            "filename"
        ]
        .astype(str)
        .to_numpy()
    ),
    selection_writers=(
        selection_dino_s_rows[
            "writer"
        ]
        .to_numpy(
            dtype=np.int64
        )
    ),
    selection_page_ids=(
        selection_dino_s_rows[
            "page_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    ),
)

with open(
    REPORT_DIR
    / "dinov2s_transformation_audit.json",
    "w",
) as file:
    json.dump(
        dino_s_transformation_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        dino_s_transformation_audit,
        indent=2,
    )
)

if (
    fit_dino_s_pca.shape
    != (
        580,
        225,
    )
    or selection_dino_s_pca.shape
    != (
        144,
        225,
    )
    or fit_dino_s_lda_raw.shape
    != (
        580,
        144,
    )
    or selection_dino_s_lda_raw.shape
    != (
        144,
        144,
    )
    or not np.isfinite(
        fit_dino_s_adapted
    ).all()
    or not np.isfinite(
        selection_dino_s_adapted
    ).all()
    or not np.allclose(
        fit_adapted_norms,
        1.0,
        atol=1e-5,
    )
    or not np.allclose(
        selection_adapted_norms,
        1.0,
        atol=1e-5,
    )
):
    raise RuntimeError(
        "DINOv2-S fit-only PCA/LDA transformation audit failed."
    )

{
  "representation": "DINOv2-S/14-Reg",
  "raw_dimension": 384,
  "raw_l2_normalization_before_pca": false,
  "pca_components": 225,
  "pca_solver": "full",
  "pca_fit_pages": 580,
  "pca_fit_writers": 145,
  "pca_selection_used_for_fit": false,
  "pca_retained_variance": 0.9965361952781677,
  "fit_pca_shape": [
    580,
    225
  ],
  "selection_pca_shape": [
    144,
    225
  ],
  "lda_solver": "svd",
  "lda_fit_pages": 580,
  "lda_fit_writers": 145,
  "lda_selection_used_for_fit": false,
  "lda_output_dimension": 144,
  "fit_lda_shape": [
    580,
    144
  ],
  "selection_lda_shape": [
    144,
    144
  ],
  "post_lda_l2_normalization": true,
  "fit_adapted_shape": [
    580,
    144
  ],
  "selection_adapted_shape": [
    144,
    144
  ],
  "fit_norm_min": 0.9999998807907104,
  "fit_norm_max": 1.0000001192092896,
  "selection_norm_min": 0.9999998807907104,
  "selection_norm_max": 1.0000001192092896,
  "selection_labels_used_for_transformation_fit": false,
  "selection_evaluate

In [7]:
SELECTION_PAIR_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "selection_cross_script_pairs.csv"
)

WGC_CROSS_MACRO_AUC = (
    0.7827656525573192
)

MATCHED_CONTROL_CROSS_MACRO_AUC = (
    0.7775848765432098
)

selection_pair_df = pd.read_csv(
    SELECTION_PAIR_PATH
)

required_pair_columns = {
    "condition",
    "left_filename",
    "right_filename",
    "label",
}

if not required_pair_columns.issubset(
    selection_pair_df.columns
):
    raise RuntimeError(
        "Unexpected selection-pair schema."
    )


def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = (
        1.0
        - tpr
    )

    difference = (
        fpr
        - fnr
    )

    crossing_indices = np.where(
        np.diff(
            np.sign(
                difference
            )
        )
        != 0
    )[0]

    if len(
        crossing_indices
    ) == 0:
        nearest_index = int(
            np.argmin(
                np.abs(
                    difference
                )
            )
        )

        eer = (
            fpr[
                nearest_index
            ]
            + fnr[
                nearest_index
            ]
        ) / 2.0

        threshold = (
            thresholds[
                nearest_index
            ]
        )

        return (
            float(
                eer
            ),
            float(
                threshold
            ),
        )

    index = int(
        crossing_indices[
            0
        ]
    )

    difference_left = (
        difference[
            index
        ]
    )

    difference_right = (
        difference[
            index
            + 1
        ]
    )

    interpolation_weight = (
        -difference_left
        / (
            difference_right
            - difference_left
        )
    )

    eer = (
        fpr[
            index
        ]
        + interpolation_weight
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    threshold = (
        thresholds[
            index
        ]
        + interpolation_weight
        * (
            thresholds[
                index
                + 1
            ]
            - thresholds[
                index
            ]
        )
    )

    return (
        float(
            eer
        ),
        float(
            threshold
        ),
    )


selection_dino_s_embedding_map = {
    filename: embedding
    for filename, embedding in zip(
        selection_dino_s_rows[
            "filename"
        ]
        .astype(str),
        selection_dino_s_adapted,
    )
}

missing_left = [
    filename
    for filename in (
        selection_pair_df[
            "left_filename"
        ].astype(str)
    )
    if filename
    not in selection_dino_s_embedding_map
]

missing_right = [
    filename
    for filename in (
        selection_pair_df[
            "right_filename"
        ].astype(str)
    )
    if filename
    not in selection_dino_s_embedding_map
]

if (
    len(
        missing_left
    ) > 0
    or len(
        missing_right
    ) > 0
):
    raise RuntimeError(
        "Selection pair filenames are missing from DINOv2-S embeddings."
    )

left_embeddings = np.stack(
    [
        selection_dino_s_embedding_map[
            filename
        ]
        for filename in (
            selection_pair_df[
                "left_filename"
            ].astype(str)
        )
    ]
)

right_embeddings = np.stack(
    [
        selection_dino_s_embedding_map[
            filename
        ]
        for filename in (
            selection_pair_df[
                "right_filename"
            ].astype(str)
        )
    ]
)

selection_labels = (
    selection_pair_df[
        "label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

selection_scores = np.sum(
    left_embeddings
    * right_embeddings,
    axis=1,
)

selection_scored_pairs = (
    selection_pair_df
    .copy()
)

selection_scored_pairs[
    "score"
] = selection_scores

pooled_auc = float(
    roc_auc_score(
        selection_labels,
        selection_scores,
    )
)

pooled_eer, pooled_eer_threshold = (
    calculate_interpolated_eer(
        selection_labels,
        selection_scores,
    )
)

condition_rows = []

for condition in sorted(
    selection_scored_pairs[
        "condition"
    ].unique()
):
    condition_df = (
        selection_scored_pairs[
            selection_scored_pairs[
                "condition"
            ] == condition
        ]
    )

    labels = (
        condition_df[
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    scores = (
        condition_df[
            "score"
        ]
        .to_numpy()
    )

    condition_auc = float(
        roc_auc_score(
            labels,
            scores,
        )
    )

    condition_eer, condition_eer_threshold = (
        calculate_interpolated_eer(
            labels,
            scores,
        )
    )

    condition_rows.append(
        {
            "condition": (
                condition
            ),
            "auc": float(
                condition_auc
            ),
            "eer": float(
                condition_eer
            ),
            "eer_threshold": float(
                condition_eer_threshold
            ),
            "pairs": int(
                len(
                    condition_df
                )
            ),
            "genuine_pairs": int(
                condition_df[
                    "label"
                ].sum()
            ),
            "impostor_pairs": int(
                (
                    condition_df[
                        "label"
                    ] == 0
                ).sum()
            ),
        }
    )

dino_s_condition_df = pd.DataFrame(
    condition_rows
)

cross_macro_auc = float(
    dino_s_condition_df[
        "auc"
    ].mean()
)

cross_macro_eer = float(
    dino_s_condition_df[
        "eer"
    ].mean()
)

dino_s_gain_vs_wgc = float(
    cross_macro_auc
    - WGC_CROSS_MACRO_AUC
)

dino_s_gain_vs_control = float(
    cross_macro_auc
    - MATCHED_CONTROL_CROSS_MACRO_AUC
)

dino_s_selection_result = {
    "representation": "DINOv2-S/14-Reg",
    "pipeline": (
        "raw 384-D -> fit-only PCA-225 -> "
        "fit-only writer-LDA -> L2 -> cosine"
    ),
    "fit_writers": int(
        fit_dino_s_rows[
            "writer"
        ].nunique()
    ),
    "selection_writers": int(
        selection_dino_s_rows[
            "writer"
        ].nunique()
    ),
    "selection_pages": int(
        len(
            selection_dino_s_rows
        )
    ),
    "selection_pairs": int(
        len(
            selection_pair_df
        )
    ),
    "cross_conditions": int(
        dino_s_condition_df[
            "condition"
        ].nunique()
    ),
    "cross_macro_auc": float(
        cross_macro_auc
    ),
    "cross_macro_eer": float(
        cross_macro_eer
    ),
    "pooled_auc": float(
        pooled_auc
    ),
    "pooled_eer": float(
        pooled_eer
    ),
    "pooled_eer_threshold": float(
        pooled_eer_threshold
    ),
    "matched_control_cross_macro_auc": float(
        MATCHED_CONTROL_CROSS_MACRO_AUC
    ),
    "wgc_cross_macro_auc": float(
        WGC_CROSS_MACRO_AUC
    ),
    "gain_vs_matched_control": float(
        dino_s_gain_vs_control
    ),
    "gain_vs_wgc": float(
        dino_s_gain_vs_wgc
    ),
    "missing_left_filenames": int(
        len(
            missing_left
        )
    ),
    "missing_right_filenames": int(
        len(
            missing_right
        )
    ),
    "selection_used_for_pca_fit": False,
    "selection_used_for_lda_fit": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

selection_scored_pairs.to_csv(
    REPORT_DIR
    / "dinov2s_selection_scored_pairs.csv",
    index=False,
)

dino_s_condition_df.to_csv(
    REPORT_DIR
    / "dinov2s_selection_condition_results.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_selection_result.json",
    "w",
) as file:
    json.dump(
        dino_s_selection_result,
        file,
        indent=2,
    )

print(
    json.dumps(
        dino_s_selection_result,
        indent=2,
    )
)

print(
    "\nCondition results:"
)

print(
    dino_s_condition_df[
        [
            "condition",
            "auc",
            "eer",
            "pairs",
            "genuine_pairs",
            "impostor_pairs",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)

if (
    len(
        selection_pair_df
    ) != 5184
    or dino_s_condition_df[
        "condition"
    ].nunique()
    != 4
    or not (
        dino_s_condition_df[
            "pairs"
        ]
        == 1296
    ).all()
    or not (
        dino_s_condition_df[
            "genuine_pairs"
        ]
        == 36
    ).all()
    or not (
        dino_s_condition_df[
            "impostor_pairs"
        ]
        == 1260
    ).all()
    or len(
        missing_left
    ) != 0
    or len(
        missing_right
    ) != 0
    or not np.isfinite(
        selection_scores
    ).all()
):
    raise RuntimeError(
        "DINOv2-S clean selection evaluation failed."
    )

{
  "representation": "DINOv2-S/14-Reg",
  "pipeline": "raw 384-D -> fit-only PCA-225 -> fit-only writer-LDA -> L2 -> cosine",
  "fit_writers": 145,
  "selection_writers": 36,
  "selection_pages": 144,
  "selection_pairs": 5184,
  "cross_conditions": 4,
  "cross_macro_auc": 0.8430555555555557,
  "cross_macro_eer": 0.22698412698412698,
  "pooled_auc": 0.8429232804232805,
  "pooled_eer": 0.22916666666666663,
  "pooled_eer_threshold": 0.08420416206941693,
  "matched_control_cross_macro_auc": 0.7775848765432098,
  "wgc_cross_macro_auc": 0.7827656525573192,
  "gain_vs_matched_control": 0.06547067901234582,
  "gain_vs_wgc": 0.060289902998236444,
  "missing_left_filenames": 0,
  "missing_right_filenames": 0,
  "selection_used_for_pca_fit": false,
  "selection_used_for_lda_fit": false,
  "monitor_used": false,
  "validation_used": false,
  "official_test_used": false
}

Condition results:
              condition      auc      eer  pairs  genuine_pairs  impostor_pairs
        cross_same_same 0.

In [8]:
wgc_condition_auc_map = {
    "cross_same_same": 0.825617,
    "cross_same_variable": 0.777976,
    "cross_variable_same": 0.738382,
    "cross_variable_variable": 0.789087,
}

dino_s_condition_auc_map = {
    str(
        row[
            "condition"
        ]
    ): float(
        row[
            "auc"
        ]
    )
    for _, row in (
        dino_s_condition_df
        .iterrows()
    )
}

condition_comparison_rows = []

for condition in sorted(
    wgc_condition_auc_map
):
    wgc_auc = float(
        wgc_condition_auc_map[
            condition
        ]
    )

    dino_s_auc = float(
        dino_s_condition_auc_map[
            condition
        ]
    )

    condition_comparison_rows.append(
        {
            "condition": (
                condition
            ),
            "wgc_auc": float(
                wgc_auc
            ),
            "dinov2s_auc": float(
                dino_s_auc
            ),
            "dinov2s_minus_wgc": float(
                dino_s_auc
                - wgc_auc
            ),
        }
    )

dinov2s_vs_wgc_condition_df = pd.DataFrame(
    condition_comparison_rows
)

WGC_POOLED_EER = (
    0.2998015873015873
)

dinov2s_pooled_eer_change_vs_wgc = float(
    dino_s_selection_result[
        "pooled_eer"
    ]
    - WGC_POOLED_EER
)

all_conditions_improved = bool(
    (
        dinov2s_vs_wgc_condition_df[
            "dinov2s_minus_wgc"
        ]
        > 0.0
    ).all()
)

minimum_condition_gain = float(
    dinov2s_vs_wgc_condition_df[
        "dinov2s_minus_wgc"
    ].min()
)

maximum_condition_gain = float(
    dinov2s_vs_wgc_condition_df[
        "dinov2s_minus_wgc"
    ].max()
)

REPRESENTATION_GAIN_THRESHOLD = (
    0.03
)

representation_advantage_supported = bool(
    dino_s_selection_result[
        "gain_vs_wgc"
    ]
    >= REPRESENTATION_GAIN_THRESHOLD
    and all_conditions_improved
)

dinov2s_representation_verdict = {
    "representation": (
        "DINOv2-S/14-Reg"
    ),
    "evaluation_role": (
        "clean representation audit"
    ),
    "pipeline": (
        "raw 384-D -> fit-only PCA-225 -> "
        "fit-only writer-LDA -> L2 -> cosine"
    ),
    "matched_control_cross_macro_auc": float(
        MATCHED_CONTROL_CROSS_MACRO_AUC
    ),
    "wgc_cross_macro_auc": float(
        WGC_CROSS_MACRO_AUC
    ),
    "dinov2s_cross_macro_auc": float(
        dino_s_selection_result[
            "cross_macro_auc"
        ]
    ),
    "dinov2s_gain_vs_matched_control": float(
        dino_s_selection_result[
            "gain_vs_matched_control"
        ]
    ),
    "dinov2s_gain_vs_wgc": float(
        dino_s_selection_result[
            "gain_vs_wgc"
        ]
    ),
    "wgc_pooled_eer": float(
        WGC_POOLED_EER
    ),
    "dinov2s_pooled_eer": float(
        dino_s_selection_result[
            "pooled_eer"
        ]
    ),
    "dinov2s_pooled_eer_change_vs_wgc": float(
        dinov2s_pooled_eer_change_vs_wgc
    ),
    "conditions_improved_vs_wgc": int(
        (
            dinov2s_vs_wgc_condition_df[
                "dinov2s_minus_wgc"
            ]
            > 0.0
        ).sum()
    ),
    "cross_conditions": int(
        len(
            dinov2s_vs_wgc_condition_df
        )
    ),
    "all_conditions_improved_vs_wgc": bool(
        all_conditions_improved
    ),
    "minimum_condition_auc_gain_vs_wgc": float(
        minimum_condition_gain
    ),
    "maximum_condition_auc_gain_vs_wgc": float(
        maximum_condition_gain
    ),
    "representation_gain_threshold": float(
        REPRESENTATION_GAIN_THRESHOLD
    ),
    "representation_advantage_supported": bool(
        representation_advantage_supported
    ),
    "resnet_representation_bottleneck_evidence": (
        "strong"
        if representation_advantage_supported
        else "insufficient"
    ),
    "next_backbone_candidate": (
        "DINOv2-S/14-Reg"
        if representation_advantage_supported
        else None
    ),
    "next_experiment": (
        "DINOv2-S student baseline"
        if representation_advantage_supported
        else "reconsider representation hypothesis"
    ),
    "pca_dimension_tuned_on_selection": False,
    "lda_configuration_tuned_on_selection": False,
    "selection_used_for_transformation_fit": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

dinov2s_vs_wgc_condition_df.to_csv(
    REPORT_DIR
    / "dinov2s_vs_wgc_condition_comparison.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_representation_verdict.json",
    "w",
) as file:
    json.dump(
        dinov2s_representation_verdict,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_representation_verdict,
        indent=2,
    )
)

print(
    "\nCondition comparison:"
)

print(
    dinov2s_vs_wgc_condition_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)

if (
    not all_conditions_improved
    or not representation_advantage_supported
):
    raise RuntimeError(
        "DINOv2-S did not satisfy the frozen representation-advantage criterion."
    )

{
  "representation": "DINOv2-S/14-Reg",
  "evaluation_role": "clean representation audit",
  "pipeline": "raw 384-D -> fit-only PCA-225 -> fit-only writer-LDA -> L2 -> cosine",
  "matched_control_cross_macro_auc": 0.7775848765432098,
  "wgc_cross_macro_auc": 0.7827656525573192,
  "dinov2s_cross_macro_auc": 0.8430555555555557,
  "dinov2s_gain_vs_matched_control": 0.06547067901234582,
  "dinov2s_gain_vs_wgc": 0.060289902998236444,
  "wgc_pooled_eer": 0.2998015873015873,
  "dinov2s_pooled_eer": 0.22916666666666663,
  "dinov2s_pooled_eer_change_vs_wgc": -0.07063492063492066,
  "conditions_improved_vs_wgc": 4,
  "cross_conditions": 4,
  "all_conditions_improved_vs_wgc": true,
  "minimum_condition_auc_gain_vs_wgc": 0.02592576014109349,
  "maximum_condition_auc_gain_vs_wgc": 0.08108484656084658,
  "representation_gain_threshold": 0.03,
  "representation_advantage_supported": true,
  "resnet_representation_bottleneck_evidence": "strong",
  "next_backbone_candidate": "DINOv2-S/14-Reg",
  "next

## Final Summary

This notebook tested whether the limited performance of the previous ResNet18-based student was primarily caused by the quality of its learned representation.

A frozen DINOv2-S representation was evaluated using the clean 145-writer fit and 36-writer selection protocol.

The representation pipeline was:

**cached normalized DINOv2-S 384-D feature → PCA(225) → writer-LDA → L2 normalization → cosine verification**

The cached DINOv2-S features had already been L2-normalized immediately after backbone extraction in the original feature-extraction pipeline. PCA was therefore applied to these pre-normalized cached features. Writer-LDA was then fitted using only the 145 development-fit writers, and the resulting LDA embeddings were L2-normalized again before cosine verification.

No transformation was fitted using the 36 selection writers.

### Selection Performance

The DINOv2-S PCA/LDA representation achieved:

- Cross-script macro AUC: **0.84306**
- Cross-script macro EER: **0.22698**
- Pooled AUC: **0.84292**
- Pooled EER: **0.22917**

The four cross-script conditions were:

| Condition | AUC | EER |
|---|---:|---:|
| cross_same_same | 0.89852 | 0.15635 |
| cross_same_variable | 0.85906 | 0.22222 |
| cross_variable_same | 0.76431 | 0.30238 |
| cross_variable_variable | 0.85033 | 0.22698 |

### Comparison with Previous Students

The previous ResNet18 WGC reference achieved a cross-script macro AUC of:

**0.78277**

The DINOv2-S PCA/LDA representation achieved:

**0.84306**

This corresponds to an absolute improvement of approximately:

**+0.06029 macro AUC**

All four cross-script conditions improved relative to the WGC reference.

The pooled EER also decreased from approximately:

**0.29980 → 0.22917**

which is an absolute reduction of approximately:

**0.07063**

### Scientific Interpretation

The result provides strong evidence that the previous ResNet18 student was substantially **representation-limited**.

Earlier experiments attempted to improve cross-script invariance and teacher transfer while retaining the same underlying ResNet18 representation. Those methods produced comparatively small or inconsistent improvements.

In contrast, replacing only the underlying representation with frozen DINOv2-S features produced a much larger improvement under the same writer-disjoint evaluation setting.

This indicates that useful cross-script writer information is already strongly encoded in DINOv2-S features and that the downstream student should first exploit this stronger representation before introducing additional complex distillation or invariance objectives.

The result therefore supports a representation pivot from the previous ResNet18 student toward DINOv2-S.

### Important Scope

This PCA/LDA result is a **representation audit**, not the final student model.

PCA and writer-LDA are supervised post-processing components fitted using the 145 development-fit writers. Therefore, the resulting 0.84306 macro AUC should be treated as evidence of the quality and accessibility of the DINOv2-S representation rather than as the performance of a lightweight end-to-end trainable student.

The appropriate next experiment is consequently a controlled DINOv2-S student in which the backbone representation remains frozen and a small trainable projection head replaces the PCA/LDA adaptation pipeline.

### Protocol Status

- DINOv2-S source features were cached and already L2-normalized: **Yes**
- PCA fitted only on development-fit writers: **Yes**
- Writer-LDA fitted only on development-fit writers: **Yes**
- Post-LDA L2 normalization applied before cosine scoring: **Yes**
- Fit and selection writers disjoint: **Yes**
- Monitor used: **No**
- Validation used: **No**
- Official test used: **No**

The DINOv2-S clean representation audit therefore provides strong support for the **representation-bottleneck hypothesis** and justifies moving to a lightweight DINOv2-S-based student.